# Laboratorio 2 desarrollado - Spark vs DuckDB vs Polars

**Asignatura:** Análisis de Big Data · Magíster en Data Science UDD  
**Estudiante:** Dante Aguirre Bravo  
**Entorno de ejecución:** VSC  
**Dataset:** NYC Yellow Taxi 2023, 12 archivos Parquet (~38 millones de viajes)

## Objetivo

Ejecutar la misma consulta analítica con funciones de ventana en Spark, DuckDB y Polars; comprobar que los resultados sean equivalentes; interpretar los planes de ejecución y recomendar un motor usando tiempos medidos.

> **Instrucción de entrega:** en Colab seleccione **Entorno de ejecución > Ejecutar todas**. La descarga completa ocupa cerca de 700 MB y puede tardar varios minutos. No cambie `MESES = 12` para la corrida final.

## 0. Preparación del entorno

Se fijan versiones solo donde el curso lo requiere. Después de instalar, Colab puede mostrar un aviso para reiniciar el entorno; normalmente no es necesario para estas bibliotecas.

In [ ]:
%pip install -q pyspark==3.5.4 duckdb polars

In [ ]:
from pathlib import Path
from functools import reduce
from IPython.display import display, Markdown
import gc
import os
import platform
import re
import statistics
import time
import urllib.request

import pandas as pd
import matplotlib.pyplot as plt

MESES = 12                 # Entrega final: mantener en 12
CORRIDAS = 3
BASE_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data"
EN_COLAB = "google.colab" in __import__("sys").modules
RUTA_DATOS = Path("/content/nyc_taxi_2023" if EN_COLAB else "./nyc_taxi_2023")
RUTA_DATOS.mkdir(parents=True, exist_ok=True)

print(f"Python: {platform.python_version()} | Colab: {EN_COLAB}")
print(f"Directorio de datos: {RUTA_DATOS.resolve()}")

In [ ]:
# Descarga reanudable y con validación mínima del tamaño de cada archivo.
for mes in range(1, MESES + 1):
    nombre = f"yellow_tripdata_2023-{mes:02d}.parquet"
    destino = RUTA_DATOS / nombre
    temporal = destino.with_suffix(".parquet.part")
    if destino.exists() and destino.stat().st_size > 1_000_000:
        print("ya existe", nombre)
        continue

    url = f"{BASE_URL}/{nombre}"
    for intento in range(1, 4):
        try:
            urllib.request.urlretrieve(url, temporal)
            if temporal.stat().st_size <= 1_000_000:
                raise IOError("descarga incompleta")
            temporal.replace(destino)
            print("ok", nombre)
            break
        except Exception as exc:
            if intento == 3:
                raise RuntimeError(f"No fue posible descargar {nombre}") from exc
            print(f"reintento {intento}/3 para {nombre}: {exc}")
            time.sleep(2 * intento)

ARCHIVOS = sorted(str(p) for p in RUTA_DATOS.glob("yellow_tripdata_2023-*.parquet"))
assert len(ARCHIVOS) == MESES, f"Se esperaban {MESES} archivos y hay {len(ARCHIVOS)}"
TAMANO_MB = sum(Path(a).stat().st_size for a in ARCHIVOS) / 1e6
print(f"{len(ARCHIVOS)} archivos · {TAMANO_MB:,.1f} MB")

In [ ]:
# El número de filas se obtiene desde el metadata de Parquet; no se escanean columnas
# ni se calienta injustamente el caché de uno de los tres motores.
import pyarrow.parquet as pq

filas_por_mes = {
    Path(a).stem[-2:]: pq.ParquetFile(a).metadata.num_rows
    for a in ARCHIVOS
}
TOTAL_FILAS = sum(filas_por_mes.values())
display(pd.DataFrame.from_dict(filas_por_mes, orient="index", columns=["filas"]).rename_axis("mes"))
print(f"Total: {TOTAL_FILAS:,} filas")

## 1. Consulta de referencia

**Pregunta de negocio:** por zona de origen, identificar los tres días de mayor demanda del año y calcular, para cada uno de esos días, la media móvil de viajes de los últimos siete días.

La lógica, los filtros, las columnas y los límites temporales son iguales en los tres motores. Se agregó `fecha ASC` como desempate explícito: así `ROW_NUMBER` es determinista cuando dos días tienen la misma cantidad de viajes.

In [ ]:
SQL = """
WITH viajes_diarios AS (
  SELECT PULocationID AS zona,
         CAST(tpep_pickup_datetime AS DATE) AS fecha,
         COUNT(*) AS viajes
  FROM viajes
  WHERE total_amount > 0
    AND trip_distance > 0
    AND tpep_pickup_datetime >= TIMESTAMP '2023-01-01 00:00:00'
    AND tpep_pickup_datetime <  TIMESTAMP '2024-01-01 00:00:00'
  GROUP BY PULocationID, CAST(tpep_pickup_datetime AS DATE)
)
SELECT zona, fecha, viajes, pos, media_movil_7d
FROM (
  SELECT zona, fecha, viajes,
         ROW_NUMBER() OVER (
           PARTITION BY zona ORDER BY viajes DESC, fecha ASC
         ) AS pos,
         ROUND(AVG(viajes) OVER (
           PARTITION BY zona ORDER BY fecha
           ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
         ), 1) AS media_movil_7d
  FROM viajes_diarios
)
WHERE pos <= 3
ORDER BY zona, pos
"""
print(SQL)

In [ ]:
resultados_tiempo = {}

def cronometrar(motor, funcion, corridas=CORRIDAS):
    """Materializa el resultado en cada corrida y conserva todos los tiempos."""
    tiempos, ultimo = [], None
    for numero in range(1, corridas + 1):
        gc.collect()
        inicio = time.perf_counter()
        ultimo = funcion()
        segundos = time.perf_counter() - inicio
        tiempos.append(segundos)
        print(f"{motor:7s} · corrida {numero}: {segundos:.2f} s")
    resultados_tiempo[motor] = {
        "primera_s": tiempos[0],
        "mediana_s": statistics.median(tiempos),
        "min_s": min(tiempos),
        "corridas_s": tiempos,
    }
    return ultimo

## 2. Motor 1: Spark

Los archivos mensuales no siempre tienen exactamente el mismo esquema. Se lee cada archivo por separado, se seleccionan únicamente las cuatro columnas necesarias y se unifican sus tipos antes del `UNION`. Esto evita que diferencias como `Airport_fee`/`airport_fee` afecten la consulta.

In [ ]:
from pyspark.sql import SparkSession, functions as F

inicio_spark = time.perf_counter()
spark = (
    SparkSession.builder
    .appName("ADBD-Lab2-Osvaldo-Neira")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)
arranque_spark_s = time.perf_counter() - inicio_spark
spark.sparkContext.setLogLevel("WARN")

columnas = {
    "PULocationID": "bigint",
    "tpep_pickup_datetime": "timestamp",
    "total_amount": "double",
    "trip_distance": "double",
}
partes = [
    spark.read.parquet(archivo).select(
        *[F.col(nombre).cast(tipo).alias(nombre) for nombre, tipo in columnas.items()]
    )
    for archivo in ARCHIVOS
]
viajes_spark = reduce(lambda izquierda, derecha: izquierda.unionByName(derecha), partes)
viajes_spark.createOrReplaceTempView("viajes")

print(f"Spark {spark.version} · arranque de sesión: {arranque_spark_s:.2f} s")
print("Columnas expuestas a la consulta:", viajes_spark.columns)

In [ ]:
def ejecutar_spark():
    return [tuple(fila) for fila in spark.sql(SQL).collect()]

resultado_spark = cronometrar("Spark", ejecutar_spark)
display(pd.DataFrame(resultado_spark, columns=["zona", "fecha", "viajes", "pos", "media_movil_7d"]).head(6))

In [ ]:
# Se materializa este mismo DataFrame antes de pedir el plan ejecutado para que AQE
# tenga información real de los shuffles y pueda mostrar el plan final adaptado.
df_plan_spark = spark.sql(SQL)
_ = df_plan_spark.collect()
plan_spark = spark._sc._jvm.PythonSQLUtils.explainString(
    df_plan_spark._jdf.queryExecution(), "formatted"
)
print(plan_spark)

### 2.1 Interpretación del plan de Spark

## Respuesta

1. **PushedFilters.** En el plan deberían aparecer los filtros de valores no nulos, `total_amount > 0`, `trip_distance > 0` y el rango de fechas junto al `FileScan parquet`. Esto significa que Catalyst llevó las condiciones hasta el lector de Parquet, permitiendo descartar datos antes de procesarlos completamente. Si algún filtro no aparece en `PushedFilters`, normalmente se debe a que una conversión de tipo impide aplicarlo directamente sobre el archivo.

2. **ReadSchema.** Spark lee solamente las cuatro columnas necesarias: `PULocationID`, `tpep_pickup_datetime`, `total_amount` y `trip_distance`, en lugar de cargar todas las columnas disponibles. Esto corresponde a la poda de columnas y reduce la lectura de información que no se utiliza en la consulta.

3. **Exchange.** El plan requiere tres redistribuciones principales. La primera permite completar la agregación diaria por zona y fecha. La segunda reúne los registros de cada zona antes de calcular las funciones de ventana. La tercera distribuye los datos para realizar el ordenamiento final por `zona` y `pos`. Los nodos `Sort` ordenan los registros dentro de las particiones, pero no representan por sí mismos un nuevo *shuffle*.

4. **AQE.** La presencia de `AdaptiveSparkPlan` indica que Adaptive Query Execution está habilitado. Después de ejecutar la acción, Spark puede ajustar el plan con las estadísticas reales, por ejemplo, uniendo particiones pequeñas de *shuffle*. Como esta consulta no contiene un `JOIN`, no corresponde esperar una conversión a `BroadcastHashJoin`.

En resumen, Spark aprovecha el pushdown de filtros y la poda de columnas para reducir la lectura, pero debe realizar los `Exchange` y ordenamientos necesarios para resolver el `GROUP BY`, las ventanas y el orden final.


## 3. Motor 2: DuckDB

DuckDB consulta los Parquet directamente desde el proceso de Python. `union_by_name=true` tolera diferencias de esquema entre meses.

In [ ]:
import duckdb

con = duckdb.connect()
patron = str(RUTA_DATOS / "yellow_tripdata_2023-*.parquet").replace("'", "''")
con.execute(f"CREATE OR REPLACE VIEW viajes AS SELECT * FROM read_parquet('{patron}', union_by_name=true)")

def ejecutar_duckdb():
    return con.execute(SQL).fetchall()

resultado_duckdb = cronometrar("DuckDB", ejecutar_duckdb)
print("DuckDB", duckdb.__version__)
display(pd.DataFrame(resultado_duckdb, columns=["zona", "fecha", "viajes", "pos", "media_movil_7d"]).head(6))

In [ ]:
plan_duckdb = con.execute("EXPLAIN " + SQL).fetchone()[1]
print(plan_duckdb)

### 3.1 Interpretación del plan de DuckDB

## Respuesta

En el plan de DuckDB, `PARQUET_SCAN` debería mostrar que se proyectan solamente las cuatro columnas necesarias y que los filtros se aplican cerca del escaneo. Esto corresponde a las mismas ideas de poda de columnas y *filter pushdown* observadas en Spark.

Después del escaneo aparecen las operaciones de agregación, ventana, filtro y ordenamiento necesarias para responder la consulta. Estas operaciones no se pueden eliminar porque forman parte de la lógica solicitada.

No aparece un nodo `Exchange` porque DuckDB se ejecuta dentro de un solo proceso y utiliza varios hilos de la misma máquina. Por lo tanto, no necesita redistribuir datos a través de una red de executors como Spark. Esto reduce el costo de coordinación cuando el dataset cabe cómodamente en un solo nodo.

En resumen, DuckDB aplica filtros y poda de columnas directamente sobre los archivos Parquet y evita el costo de una ejecución distribuida, aunque su capacidad queda limitada por los recursos de una sola máquina.


## 4. Motor 3: Polars

Se usa la API *lazy*: `scan_parquet` construye el plan, y solo `collect()` lo materializa. Cada archivo se proyecta a las mismas cuatro columnas y tipos.

In [ ]:
import polars as pl

def scan_viajes():
    return pl.concat([
        pl.scan_parquet(archivo).select(
            pl.col("PULocationID").cast(pl.Int64),
            pl.col("tpep_pickup_datetime").cast(pl.Datetime("us")),
            pl.col("total_amount").cast(pl.Float64),
            pl.col("trip_distance").cast(pl.Float64),
        )
        for archivo in ARCHIVOS
    ])

def consulta_polars_lazy():
    diarios = (
        scan_viajes()
        .filter(
            (pl.col("total_amount") > 0)
            & (pl.col("trip_distance") > 0)
            & (pl.col("tpep_pickup_datetime") >= pl.datetime(2023, 1, 1))
            & (pl.col("tpep_pickup_datetime") < pl.datetime(2024, 1, 1))
        )
        .group_by(
            zona=pl.col("PULocationID"),
            fecha=pl.col("tpep_pickup_datetime").dt.date(),
        )
        .agg(viajes=pl.len())
        .sort(["zona", "fecha"])
    )
    return (
        diarios
        .with_columns(
            pos=pl.col("viajes").rank("ordinal", descending=True).over("zona"),
            media_movil_7d=(
                pl.col("viajes")
                .rolling_mean(window_size=7, min_samples=1)
                .over("zona")
                .round(1)
            ),
        )
        .filter(pl.col("pos") <= 3)
        .select("zona", "fecha", "viajes", "pos", "media_movil_7d")
        .sort(["zona", "pos"])
    )

def ejecutar_polars():
    return consulta_polars_lazy().collect()

resultado_polars = cronometrar("Polars", ejecutar_polars)
print("Polars", pl.__version__)
print(resultado_polars.head(6))

In [ ]:
plan_polars = consulta_polars_lazy().explain()
print(plan_polars)

### 4.1 Interpretación del plan de Polars

## Respuesta

Polars utiliza la API *lazy*, por lo que `scan_parquet` construye el plan y `collect()` inicia la ejecución. En el plan optimizado, `PROJECT` muestra que se leen solamente las cuatro columnas necesarias y `SELECTION` junto al escaneo Parquet indica que los filtros se aplican lo más cerca posible de la fuente.

Al igual que DuckDB, Polars trabaja de forma local y multihilo. Por esta razón, no aparece un `Exchange` de red como en Spark. Sin embargo, sí debe realizar agrupaciones y ordenamientos porque son necesarios para calcular la demanda diaria, el ranking y la media móvil.

La evaluación *lazy* permite que Polars analice la cadena completa de operaciones antes de materializar el resultado y aplique optimizaciones sobre el plan.

En resumen, Polars reduce la lectura mediante la poda de columnas y el pushdown de filtros, ejecuta la consulta con varios hilos en una sola máquina y evita el costo de coordinación de un clúster distribuido.


## 5. Validación de equivalencia

Un benchmark solo es válido si los tres motores producen el mismo resultado. Se normalizan tipos y orden, se compara cada celda y se revisa la zona 132 (aeropuerto JFK), sugerida por la guía.

In [ ]:
COLUMNAS_RESULTADO = ["zona", "fecha", "viajes", "pos", "media_movil_7d"]

def canonizar(df):
    copia = df.copy()
    copia["zona"] = copia["zona"].astype("int64")
    copia["fecha"] = pd.to_datetime(copia["fecha"]).dt.strftime("%Y-%m-%d")
    copia["viajes"] = copia["viajes"].astype("int64")
    copia["pos"] = copia["pos"].astype("int64")
    copia["media_movil_7d"] = copia["media_movil_7d"].astype(float).round(1)
    return copia.sort_values(["zona", "pos"]).reset_index(drop=True)

spark_pd = canonizar(pd.DataFrame(resultado_spark, columns=COLUMNAS_RESULTADO))
duckdb_pd = canonizar(pd.DataFrame(resultado_duckdb, columns=COLUMNAS_RESULTADO))
polars_pd = canonizar(pd.DataFrame(resultado_polars.rows(), columns=COLUMNAS_RESULTADO))

pd.testing.assert_frame_equal(spark_pd, duckdb_pd, check_dtype=False, rtol=0, atol=0.05)
pd.testing.assert_frame_equal(spark_pd, polars_pd, check_dtype=False, rtol=0, atol=0.05)
assert spark_pd["pos"].between(1, 3).all()
assert not spark_pd.duplicated(["zona", "pos"]).any()

print(f"VALIDACIÓN OK: {len(spark_pd):,} filas idénticas en los tres motores.")
display(spark_pd.query("zona == 132").style.hide(axis="index"))

## 6. Resultados del benchmark y recomendación

La primera corrida aproxima el escenario frío; la mediana de tres reduce el efecto de variaciones puntuales. El arranque del JVM se informa por separado y no se mezcla con el tiempo de la consulta. Los tres motores acceden a los mismos archivos locales; aun así, el orden de ejecución y el caché del sistema operativo son limitaciones de un benchmark en un único entorno Colab.

In [ ]:
tabla_tiempos = (
    pd.DataFrame(resultados_tiempo).T[["primera_s", "mediana_s", "min_s"]]
    .rename(columns={
        "primera_s": "1a corrida (s)",
        "mediana_s": "mediana 3 corridas (s)",
        "min_s": "mejor corrida (s)",
    })
)
tabla_tiempos["filas procesadas"] = TOTAL_FILAS
tabla_tiempos["datos (MB)"] = round(TAMANO_MB, 1)
display(tabla_tiempos.round(2))

ax = tabla_tiempos["mediana 3 corridas (s)"].sort_values().plot(
    kind="bar", color=["#16a085", "#2980b9", "#e67e22"], figsize=(8, 4)
)
ax.set_title("Tiempo mediano por motor - misma consulta y datos")
ax.set_xlabel("")
ax.set_ylabel("segundos (menor es mejor)")
ax.grid(axis="y", alpha=0.25)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
medianas = tabla_tiempos["mediana 3 corridas (s)"].astype(float)
ganador = medianas.idxmin()
segundo = medianas.sort_values().index[1]
ventaja = medianas[segundo] / medianas[ganador]

if ganador in {"DuckDB", "Polars"}:
    cambio = (
        "Cambiaría a Spark al crecer hacia cientos de GB/escala TB, si el pipeline "
        "ya reside en un lakehouse distribuido o si se necesita streaming distribuido."
    )
    costo_oculto = (
        "La opción ganadora depende de la RAM, CPU y disco de un solo nodo; tiene menor "
        "tolerancia a fallos y no escala horizontalmente como Spark."
    )
else:
    cambio = (
        "Cambiaría a DuckDB o Polars si los datos siguen cabiendo en un nodo y predominan "
        "consultas exploratorias, porque se evita el costo fijo del JVM y del cluster."
    )
    costo_oculto = (
        "Spark agrega arranque, operación del cluster, shuffles y costo cloud; la tabla no "
        "monetiza infraestructura ni tiempo de administración."
    )

recomendacion = f'''### Recomendación ejecutiva (generada con las mediciones de esta corrida)

1. Para **{TOTAL_FILAS:,} filas ({TAMANO_MB:,.0f} MB)** recomiendo **{ganador}**: su mediana fue **{medianas[ganador]:.2f} s**, frente a **{medianas[segundo]:.2f} s** del segundo; fue **{ventaja:.2f}x** más rápido.
2. {cambio}
3. {costo_oculto}
'''
display(Markdown(recomendacion))

### Conclusión analítica

## Respuesta

La recomendación debe basarse en la tabla de tiempos generada por esta ejecución. Para un dataset cercano a 700 MB, es esperable que DuckDB o Polars obtengan un buen resultado porque los datos caben en una sola máquina y no necesitan la coordinación distribuida que utiliza Spark. Sin embargo, el motor recomendado será el que obtenga la menor mediana en las tres corridas, no el que se prefiera antes de realizar el benchmark.

La interpretación de los planes ayuda a explicar los tiempos. Los tres motores aplican filtros y leen solamente las columnas necesarias, pero Spark además debe pagar el costo de los `Exchange`, los ordenamientos y la coordinación del JVM. En un volumen pequeño o mediano, ese costo fijo puede representar una parte importante del tiempo total.

Spark sería más conveniente cuando el volumen deje de caber en una sola máquina, el procesamiento crezca a cientos de GB o TB, los datos ya se encuentren en un *lakehouse* distribuido o se necesiten capacidades como *streaming*, tolerancia a fallos y escalamiento horizontal.

También se debe considerar que este benchmark mide principalmente tiempo de ejecución. No compara memoria máxima, concurrencia, mantenimiento, costo de infraestructura ni tolerancia a fallos, por lo que el motor más rápido en Colab no necesariamente será la mejor opción para todos los escenarios.

En resumen, para este volumen se recomienda el motor con la menor mediana observada, pero la decisión puede cambiar cuando aumentan el tamaño de los datos y los requisitos de operación distribuida.


## 7. Respuestas a las extensiones

### E1 — JOIN con nombres de zonas

Al unir los viajes con el catálogo de 265 zonas, debería aparecer `BroadcastHashJoin` si Spark identifica que la tabla de zonas es menor que `spark.sql.autoBroadcastJoinThreshold`. En este caso, Spark copia la tabla pequeña a cada executor y evita redistribuir el dataset grande para resolver el `JOIN`.

En resumen, el *broadcast join* es conveniente porque la dimensión de zonas es pequeña y puede enviarse completa a cada executor.

### E2 — Benchmark con un solo mes

Al utilizar solamente un mes, DuckDB o Polars podrían ampliar su ventaja relativa porque Spark tiene costos fijos de inicio, planificación y coordinación que se amortizan mejor con volúmenes grandes. Sin embargo, esto es una hipótesis y debe comprobarse ejecutando los tres motores con `MESES = 1` y comparando nuevamente sus medianas.

En resumen, reducir el volumen puede cambiar el orden de los motores, pero la conclusión debe basarse en una nueva medición y no solamente en lo esperado.

### E3 — Ventana sin `PARTITION BY zona`

Si se elimina `PARTITION BY zona`, Spark debe calcular una ventana global. El plan debería mostrar `Exchange SinglePartition`, porque todos los registros necesitan llegar a una sola partición para conservar un único orden.

Esto elimina gran parte del paralelismo, concentra el uso de memoria y puede producir un cuello de botella o un error por falta de memoria cuando aumenta el volumen.

En resumen, quitar `PARTITION BY zona` obliga a Spark a reunir los datos en una sola partición y transforma una operación paralela en una operación difícil de escalar.


## 8. Declaración de uso de IA

Para desarrollar este laboratorio se utilizó un asistente de inteligencia artificial como herramienta de apoyo. Su función principal fue ayudar a revisar la estructura del notebook, proponer mejoras en la implementación y entregar una primera explicación de los resultados. Las respuestas y conclusiones no se aceptaron automáticamente, sino que fueron revisadas utilizando el código, los planes de ejecución y los resultados obtenidos en el notebook.

El asistente de IA se utilizó específicamente para:

- revisar la descarga de los archivos y la normalización de los esquemas mensuales, debido a que algunas columnas pueden presentar diferencias de nombre o tipo;
- proponer una implementación equivalente de la consulta con funciones de ventana en Spark SQL, DuckDB SQL y Polars LazyFrame;
- apoyar la interpretación inicial de los planes de ejecución de los tres motores;
- mejorar la redacción de las respuestas y de la recomendación final;
- revisar que la recomendación se construyera con los tiempos medidos durante la ejecución y no con valores inventados o definidos previamente.

Después de utilizar la IA, se realizó una validación propia del trabajo. Primero, se comprobó que los tres motores respondieran la misma pregunta de negocio y utilizaran los mismos filtros, columnas y límites de fecha. También se agregó `fecha ASC` como criterio de desempate para que `ROW_NUMBER()` produzca un resultado determinista cuando dos días tienen la misma cantidad de viajes.

Luego, se normalizaron los tipos y el orden de las columnas y se compararon automáticamente todas las filas obtenidas por Spark, DuckDB y Polars. Además, se revisó de forma particular la zona 132, correspondiente al aeropuerto JFK. La ejecución solo se considera válida cuando aparece el mensaje `VALIDACIÓN OK`.

Finalmente, se revisaron los planes de ejecución para identificar la aplicación de filtros, la poda de columnas, los nodos `Exchange`, el uso de AQE y las diferencias entre una ejecución distribuida y una ejecución local. La recomendación final utiliza la mediana de las tres corridas de cada motor, por lo que el motor recomendado depende de los resultados reales obtenidos al ejecutar el notebook.

En resumen, la IA se utilizó como apoyo para revisar, implementar y redactar el laboratorio, pero los resultados fueron comprobados mediante la ejecución del código, la comparación entre los tres motores y la revisión de sus planes de ejecución.

---

**Checklist de entrega:** ejecutar todas las celdas con `MESES = 12` · verificar que aparezca `VALIDACIÓN OK` · comprobar que se impriman los planes de Spark, DuckDB y Polars · revisar que la tabla de tiempos contenga tres corridas por motor · guardar el notebook con todas sus salidas visibles.
